# 5G Network Intelligence — Past KPI Attention Results and Error Analysis

## Research question
Does a target-patch model benefit from explicit attention to supporting KPI histories? Compare two equal-capacity cases: constant covariate tokens and observed past covariates. This is a TimeXer-inspired adaptation, not a paper reproduction. See [literature and adaptations](../docs/literature_driven_training.md).

## Frozen temporal protocol
Two reused development folds: fit/stop/score endpoints 335/383/443 and 395/443/503. Context 168h; horizons 1/6/24h; all 2,880 beams. Original hours 504 onward are excluded. Global per-KPI preprocessing uses the fit prefix only. No holdout, future covariates, fitted beam summaries or beam identity embeddings.

In [ ]:
# Review only: no model import, fitting, calibration or checkpoint loading.
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
PROJECT = Path.cwd()
if not (PROJECT/'reports').is_dir() and (PROJECT.parent/'reports').is_dir():
    PROJECT = PROJECT.parent
REPORTS = PROJECT/'reports'


## Provenance and execution status
Training runs on hosted Colab T4 after cancellation of the duplicate queued Kaggle job. Weights stay on Drive. The snapshot below is dated evidence, not a live runtime monitor. An epoch log does not establish a final accuracy gain.

In [ ]:
path = REPORTS/'past_covariate_attention_progress.json'
if path.exists():
    display(json.loads(path.read_text(encoding='utf-8')))
else:
    print('No saved progress evidence.')


## Independent integrity and replay
The verifier checks pinned source/data, artifact hashes, preprocessing, cold-state equality, stopping selection, supervision and optimizer steps. It reconstructs all metrics and replays 576 history-only predictions at fixed atol 0.0001. A pending or failed audit cannot supply accepted quality evidence. Verifier: `scripts/verify_colab_past_covariates.py`.

In [ ]:
path = REPORTS/'past_covariate_attention_verified_metrics.json'
proof = json.loads(path.read_text(encoding='utf-8')) if path.exists() else None
if proof:
    display({k:v for k,v in proof.items() if k != 'metrics'})
else:
    print('Independent verification pending; no accepted quality conclusion.')


## Full-population metrics
MAE uses physical DLPRB usage units; WAPE (%) is 100 × summed absolute error / summed absolute observed load. `n_valid` is observed target coverage. Zero false-positive rate describes predictions above zero among observed zero targets. Display every case/fold/horizon; never select a favorable subset.

In [ ]:
if proof and proof.get('all_metrics_reconstructed'):
    metrics = pd.DataFrame(proof['metrics'])
    metrics['WAPE (%)'] = 100*metrics['wape']
    display(metrics.sort_values(['case','fold','horizon']))
else:
    print('Verified full-population metrics not available.')


## Paired comparisons and gates
Assess both cases against the fixed strong comparator and supporting-KPI attention against its newly trained control. Initial screen: at least 2% mean relative MAE improvement without any fold/horizon regression. A 5% point gain still needs robustness and uncertainty evidence. Gate arithmetic does not fit or calibrate a model.

In [ ]:
comparison = REPORTS/'past_covariate_attention_paired_comparisons.csv'
decision = REPORTS/'past_covariate_attention_quality_assessment.json'
if proof and comparison.exists() and decision.exists():
    display(pd.read_csv(comparison))
    display(json.loads(decision.read_text(encoding='utf-8')))
else:
    print('Complete verified comparison pending; no best-case selection.')


## Negative results and limitations
Previous Telecom data/budget and available-label studies passed integrity checks but failed quality gates. Their reports remain preserved. More data, more updates or a newer architecture do not guarantee a gain. These repeatedly reused development folds cannot establish fresh generalization. Inspect 24h degradation and zero false positives as well as average MAE. Training this architecture is still an experiment, not a production upgrade; no improved-model or WAPE-below-10% claim is made while evidence is incomplete.